# Visual Diagnostics for Stationarity

**Week 2 — Stationarity | Notebook 1 of 3**

Before running any statistical test, we start with **visual inspection**. A stationary series has:
- Constant mean over time (no trend)
- Constant variance over time (no fanning/narrowing)

Rolling mean and rolling variance plots are the primary visual tool for detecting violations.

This notebook compares a clearly **non-stationary** series (Airline Passengers) with a roughly **stationary** one (Sunspots) to build intuition for what stationarity looks like.

In [ ]:
import sys
sys.path.insert(0, "../..")

import matplotlib.pyplot as plt
import seaborn as sns

from week__01__fundamentals.ts_loader import TimeSeriesLoader
from week__02__stationarity.stationarity_tests import StationarityTester

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 4)

## 1. Load Datasets

In [ ]:
loader_airline = TimeSeriesLoader(name="AirPassengers")
airline, airline_summary = loader_airline.from_statsmodels("airline")
print(airline_summary)

loader_sunspots = TimeSeriesLoader(name="Sunspots")
sunspots, sunspots_summary = loader_sunspots.from_statsmodels("sunspots")
print(sunspots_summary)

## 2. Raw Series Plots

First, just look at the raw data. What do we see?
- **Airline Passengers**: clear upward trend + seasonal amplitude that grows with level → non-stationary
- **Sunspots**: no persistent trend, roughly constant amplitude → plausibly stationary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(airline, color="steelblue")
axes[0].set_title("Airline Passengers — clearly non-stationary")
axes[0].set_ylabel("Passengers")

axes[1].plot(sunspots, color="coral")
axes[1].set_title("Sunspots — roughly stationary")
axes[1].set_ylabel("Sunspot count")

plt.tight_layout()
plt.show()

## 3. Rolling Mean and Rolling Variance

The rolling window approach formalises the visual test:
- **Rolling mean**: if it trends up/down, the series has a non-constant mean → non-stationary
- **Rolling variance**: if it grows over time, the series has heteroscedastic variance → non-stationary

We use the `StationarityTester.rolling_diagnostics()` static method.

In [ ]:
def plot_rolling_diagnostics(series: "pd.Series", window: int, title: str) -> None:
    """Plot raw series with overlaid rolling mean and a separate rolling variance panel."""
    diag = StationarityTester.rolling_diagnostics(series, window=window)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Top: raw + rolling mean
    axes[0].plot(series, alpha=0.5, label="Raw")
    axes[0].plot(diag["rolling_mean"], color="red", linewidth=2, label=f"Rolling Mean (w={window})")
    axes[0].set_title(f"{title} — Rolling Mean")
    axes[0].legend()

    # Bottom: rolling variance
    axes[1].plot(diag["rolling_var"], color="darkorange", linewidth=2)
    axes[1].set_title(f"{title} — Rolling Variance")
    axes[1].set_ylabel("Variance")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rolling_diagnostics(airline, window=12, title="Airline Passengers")

**Interpretation — Airline Passengers:**
- Rolling mean is clearly trending upward — the mean is not constant over time.
- Rolling variance increases dramatically — the seasonal swings get wider as the level rises.
- Both violations confirm: this series is **non-stationary** and needs both a variance-stabilising transform (log) and differencing.

In [ ]:
plot_rolling_diagnostics(sunspots, window=11, title="Sunspots")

**Interpretation — Sunspots:**
- Rolling mean oscillates around a roughly constant level — no persistent trend.
- Rolling variance fluctuates but does not systematically increase or decrease.
- Visual verdict: this series is **plausibly stationary** (though the ADF/KPSS tests in Notebook 2 will confirm formally).

## 4. Key Takeaway

Visual diagnostics are the **first step**, not a replacement for formal tests. They help us:
1. Decide whether a variance-stabilising transform is needed (growing variance → yes)
2. Build intuition for what the ADF/KPSS tests should confirm
3. Catch obvious problems (structural breaks, outliers) that tests may not flag

**Next:** Notebook 2 formalises these observations with ADF and KPSS tests.